In [1]:
%run 10_MNESIS_polychronous-chains.ipynb

datetag = '2026-04-21'
SNNtorch version 0.9.4
Spikes in one target 1024.0,  in a SM window 42.0
for a value opt.lif_beta=0.7, the time constant is 2.8 steps
Spikes in one target 1024.0,  in a SM window 42.0
for a value opt.lif_beta=0.7, the time constant is 2.8 steps


## defining patterns as some Spiking Heidelberg Digits

In [2]:
opt = Params()
hd = HD_SNN(opt)
hd.net.to(hd.opt.device)

Sequential(
  (dropout): Dropout(p=0.37, inplace=False)
  (lin): Linear(in_features=41984, out_features=1024, bias=False)
  (lif): Leaky()
)

In [3]:
data_cache.joinpath('shd_data')

PosixPath('../cached_data/shd_data')

In [4]:
from pathlib import Path
import torch
import numpy as np
import h5py
import urllib.request
import ssl
import gzip
import shutil

def download_shd_dataset(data_path):
    """Download real SHD dataset from zenkelab.org"""
    data_path.mkdir(exist_ok=True, parents=True)
    
    # URLs for SHD dataset
    urls = {
        "shd_train.h5": "https://zenkelab.org/datasets/shd_train.h5.gz",
        "shd_test.h5": "https://zenkelab.org/datasets/shd_test.h5.gz"
    }
    
    downloaded_files = []
    
    # Create SSL context that ignores certificate issues (USE WITH CAUTION)
    ssl_context = ssl.create_default_context()
    ssl_context.check_hostname = False
    ssl_context.verify_mode = ssl.CERT_NONE
    
    for filename, url in urls.items():
        file_path = data_path / filename
        
        if not file_path.exists():
            gz_path = data_path / f"{filename}.gz"
            print(f"Downloading {filename}...")
            try:
                # Download with SSL context
                request = urllib.request.Request(url)
                response = urllib.request.urlopen(request, context=ssl_context)
                
                with open(gz_path, 'wb') as f:
                    f.write(response.read())
                
                # Decompress
                print(f"Decompressing {filename}...")
                with gzip.open(gz_path, 'rb') as f_in:
                    with open(file_path, 'wb') as f_out:
                        shutil.copyfileobj(f_in, f_out)
                gz_path.unlink()  # Remove compressed file
                
                print(f"Successfully downloaded {filename}")
                
            except Exception as e:
                print(f"Failed to download {filename}: {e}")
                if gz_path.exists():
                    gz_path.unlink()
                continue
        
        if file_path.exists():
            downloaded_files.append(file_path)
            print(f"Found {filename}")
    
    return downloaded_files

def load_shd_events(filepath):
    """Load SHD events from HDF5 file"""
    print(f"Loading {filepath.name}...")
    
    with h5py.File(filepath, 'r') as f:
        # Print structure for debugging
        print("HDF5 structure:")
        def print_structure(name, obj):
            print(f"  {name}: {type(obj)}")
        f.visititems(print_structure)
        
        # Load the data - SHD typically has this structure
        try:
            # Try common SHD structure
            times = f['spikes']['times'][:]
            units = f['spikes']['units'][:]
            labels = f['labels'][:]
            
            return times, units, labels
        except KeyError as e:
            print(f"KeyError: {e}")
            print("Available keys in file:", list(f.keys()))
            
            # Try alternative structures
            try:
                times = f['times'][:]
                units = f['units'][:]
                labels = f['targets'][:] if 'targets' in f else f['labels'][:]
                return times, units, labels
            except:
                raise ValueError("Could not find expected data structure in HDF5 file")

def neuron_id_to_xy(neuron_ids, width=128, height=700):
    """Convert neuron IDs to (x, y) coordinates"""
    x = neuron_ids % width
    y = neuron_ids // width
    y = np.clip(y, 0, height - 1)
    return x, y

def events_to_dense_for_sample(times, units, time_steps=100, height=700, width=128):
    """Convert SHD events for one sample to dense tensor"""
    if len(times) == 0:
        return torch.zeros(time_steps, 1, height, width)
    
    # Convert neuron IDs to coordinates
    x_coords, y_coords = neuron_id_to_xy(units, width, height)
    
    # Create time bins
    max_time = times.max() if len(times) > 0 else 1.0
    time_bins = np.linspace(0, max_time, time_steps + 1)
    
    # Initialize dense tensor
    dense_tensor = torch.zeros(time_steps, 1, height, width)
    
    # Bin events into time steps
    time_indices = np.digitize(times, time_bins) - 1
    time_indices = np.clip(time_indices, 0, time_steps - 1)
    
    # Place events in tensor
    for i in range(len(times)):
        t_idx = time_indices[i]
        x_idx = int(x_coords[i])
        y_idx = int(y_coords[i])
        
        if 0 <= x_idx < width and 0 <= y_idx < height:
            dense_tensor[t_idx, 0, y_idx, x_idx] += 1
    
    return dense_tensor

def get_real_shd_as_dense(data_path):
    """Get real SHD dataset as dense tensors"""
    # Download dataset
    files = download_shd_dataset(data_path)
    
    if not files:
        raise ValueError("No files downloaded successfully")
    
    all_dense_samples = []
    all_labels = []
    
    # Process each file
    for filepath in files[:1]:  # Start with just train file
        print(f"\nProcessing {filepath.name}...")
        try:
            times_array, units_array, labels_array = load_shd_events(filepath)
            print(f"Loaded {len(labels_array)} samples")
            
            # Process first few samples as example
            num_samples = min(5, len(labels_array))  # Limit for testing
            for i in range(num_samples):
                dense_tensor = events_to_dense_for_sample(
                    times_array[i], units_array[i], time_steps=100
                )
                all_dense_samples.append(dense_tensor)
                all_labels.append(labels_array[i])
                
                if i % 100 == 0:
                    print(f"Processed {i}/{num_samples} samples...")
                    
        except Exception as e:
            print(f"Error processing {filepath}: {e}")
            import traceback
            traceback.print_exc()
            continue
    
    if all_dense_samples:
        result = torch.stack(all_dense_samples), torch.tensor(all_labels)
        print(f"\nFinal result shape: {result[0].shape}")
        return result
    else:
        raise ValueError("No data processed successfully")

# Usage
print("Loading real SHD dataset from zenkelab.org...")
try:
    data_path = data_cache.joinpath('shd_data')
    dense_data, labels = get_real_shd_as_dense(data_path)
    
    print(f"\nResults:")
    print(f"Dense data shape: {dense_data.shape}")
    print(f"Labels shape: {labels.shape}")
    print(f"Label range: {labels.min()} to {labels.max()}")
    print(f"Sample non-zero count: {(dense_data[0] > 0).sum()}")
    
except Exception as e:
    print(f"Error loading SHD dataset: {e}")
    import traceback
    traceback.print_exc()

ModuleNotFoundError: No module named 'h5py'

In [ ]:
import os
import urllib.request
import gzip, shutil
from tensorflow.keras.utils import get_file
cache_dir=os.path.expanduser("~/data")
cache_subdir="hdspikes"
print("Using cache dir: %s"%cache_dir)
# The remote directory with the data files
base_url = "https://zenkelab.org/datasets"
# Retrieve MD5 hashes from remote
response = urllib.request.urlopen("%s/md5sums.txt"%base_url)
data = response.read() 
lines = data.decode('utf-8').split("\n")
file_hashes = { line.split()[1]:line.split()[0] for line in lines if len(line.split())==2 }
def get_and_gunzip(origin, filename, md5hash=None):
    gz_file_path = get_file(filename, origin, md5_hash=md5hash, cache_dir=cache_dir, cache_subdir=cache_subdir)
    hdf5_file_path=gz_file_path[:-3]
    if not os.path.isfile(hdf5_file_path) or os.path.getctime(gz_file_path) > os.path.getctime(hdf5_file_path):
        print("Decompressing %s"%gz_file_path)
        with gzip.open(gz_file_path, 'r') as f_in, open(hdf5_file_path, 'wb') as f_out:
            shutil.copyfileobj(f_in, f_out)
    return hdf5_file_path
# Download the Spiking Heidelberg Digits (SHD) dataset
files = [ "shd_train.h5.gz", 
          "shd_test.h5.gz",
        ]
for fn in files:
    origin = "%s/%s"%(base_url,fn)
    hdf5_file_path = get_and_gunzip(origin, fn, md5hash=file_hashes[fn])
    print(hdf5_file_path)
# Similarly, to download the SSC dataset
files = [ "ssc_train.h5.gz", 
          "ssc_valid.h5.gz",
          "ssc_test.h5.gz",
        ]
for fn in files:
    origin = "%s/%s"%(base_url,fn)
    hdf5_file_path = get_and_gunzip(origin,fn,md5hash=file_hashes[fn])
    print(hdf5_file_path)
# At this point we can visualize some of the data
import tables
import numpy as np
fileh = tables.open_file(hdf5_file_path, mode='r')
units = fileh.root.spikes.units
times = fileh.root.spikes.times
labels = fileh.root.labels
# This is how we access spikes and labels
index = 0
print("Times (ms):", times[index])
print("Unit IDs:", units[index])
print("Label:", labels[index])
# A quick raster plot for one of the samples
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(16,4))
idx = np.random.randint(len(times),size=3)
for i,k in enumerate(idx):
    ax = plt.subplot(1,3,i+1)
    ax.scatter(times[k],700-units[k], color="k", alpha=0.33, s=2)
    ax.set_title("Label %i"%labels[k])
    ax.axis("off")
plt.show()

In [ ]:
dense_data.shape

In [ ]:
labels

In [9]:
%pip install h5py

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 38.2 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [10]:
import snntorch as snn
from snntorch.spikevision import spikedata

from torch.utils.data import DataLoader

# create datasets
data_path = data_cache.joinpath('shd_data')
# train_ds = spikedata.SHD(str(data_path), train=True)
test_ds = spikedata.SHD(str(data_path), train=False)

# create dataloaders
# train_dl = DataLoader(train_ds, shuffle=True, batch_size=64)
test_dl = DataLoader(test_ds, shuffle=False, batch_size=64)

The following files do not exist, will attempt download:
../cached_data/shd_data/shd_train.hdf5
../cached_data/shd_data/shd_test.hdf5


38141952it [00:01, 31582462.37it/s]                              


Extracting ../cached_data/shd_data/shd_test.h5.gz to ../cached_data/shd_data


130842624it [00:17, 7631506.57it/s]                               


Extracting ../cached_data/shd_data/shd_train.h5.gz to ../cached_data/shd_data


/Users/laurent/sdrive_cnrs/hot_from_git/TemporallyHeterogeneousConnections/MNESIS/.venv/lib/python3.12/site-packages/snntorch/spikevision/spikedata/shd.py:121: RuntimeWarning: overflow encountered in cast
  tm = np.int32(f["spikes"]["times"][i][:] * 1e6)
/Users/laurent/sdrive_cnrs/hot_from_git/TemporallyHeterogeneousConnections/MNESIS/.venv/lib/python3.12/site-packages/snntorch/spikevision/spikedata/shd.py:121: RuntimeWarning: invalid value encountered in multiply
  tm = np.int32(f["spikes"]["times"][i][:] * 1e6)
/Users/laurent/sdrive_cnrs/hot_from_git/TemporallyHeterogeneousConnections/MNESIS/.venv/lib/python3.12/site-packages/snntorch/spikevision/spikedata/shd.py:121: RuntimeWarning: invalid value encountered in cast
  tm = np.int32(f["spikes"]["times"][i][:] * 1e6)


Creating shd.hdf5...
shd.hdf5 was created successfully.


In [ ]:
import snntorch as snn
from snntorch.spikevision import spikedata
import matplotlib.pyplot as plt
import snntorch.spikeplot as splt
import torch
from torch.utils.data import DataLoader

# Load the dataset
data_path = data_cache.joinpath('shd_data')
test_ds = spikedata.SHD(str(data_path), train=False)

# Create a dataloader
test_dl = DataLoader(test_ds, shuffle=False, batch_size=1)

# Get one batch of data
data, labels = next(iter(test_dl))

# Select the first sample from the batch
sample_data = data[0]  # This should be the spike data for one digit
sample_label = labels[0]

print(f"Sample label: {sample_label}")
print(f"Sample data shape: {sample_data.shape}")

# Plot the spike data
# The data needs to have time as the first dimension for plotting
fig, ax = plt.subplots(figsize=(10, 6))

# Assuming the data format is (time_steps, channels, height, width)
# We might need to reshape or select specific dimensions to visualize
# Let's try to plot using raster plot approach

# If sample_data is in the right format for splt.raster, use it directly
# Otherwise, we need to prepare the data appropriately
try:
    # Try using snntorch spikeplot raster function
    splt.raster(sample_data, fig, ax)
    plt.title(f"SHD Digit Sample - Label: {sample_label}")
    plt.xlabel("Time")
    plt.ylabel("Neuron ID")
    plt.show()
except Exception as e:
    print(f"Error plotting with splt.raster: {e}")
    # Alternative plotting approach
    print("Trying alternative plotting method...")
    
    # If the data is dense tensor format, we can create a simple visualization
    if len(sample_data.shape) >= 3:
        # Sum over time dimension to see spatial pattern
        spatial_pattern = sample_data.sum(dim=0)  # Sum across time steps
        plt.figure(figsize=(8, 6))
        plt.imshow(spatial_pattern.squeeze(), cmap='hot', aspect='auto')
        plt.colorbar(label='Spike Count')
        plt.title(f"SHD Digit Sample Spatial Pattern - Label: {sample_label}")
        plt.xlabel("Channel/Neuron Index")
        plt.ylabel("Time Bins")
        plt.show()

KeyError: "Unable to synchronously open object (object 'labels' doesn't exist)"

In [ ]:

# Make the target periodic
for i_SM, angle in enumerate(np.linspace(0, 2*np.pi, opt.N_SM, endpoint=False)):
    TW = get_TW_spike(angle=angle)
    hd.target[i_SM, :, :] = TW.reshape((opt.N_neuron, opt.N_time))
hd.target.shape

## learning Spiking Heidelberg Digits patterns

In [ ]:
model_filename = data_cache / f"{hd.opt.datetag}_shd.pth"
lock_filename = data_cache / model_filename.with_suffix('.lock')
# if False:
if RECOMPUTE:
    model_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
    lock_filename.unlink(missing_ok=True) # FORCING RECOMPUTE
try:
    model_state_dict = torch.load(model_filename, map_location=torch.device(hd.opt.device))
    hd.net.load_state_dict(model_state_dict)
    hd.net.eval()
    lock_filename.unlink(missing_ok=True) # in case the lock file was not unlinked
    print(f"Model weights loaded from {model_filename}") # Add a log message
except FileNotFoundError:
    if not lock_filename.exists():
        print(f"Model file not found: {model_filename}, intitializing the new model.")
        lock_filename.touch(exist_ok=True)
        ##################
        hd.update_weight()
        hd.learn_model()
        ##################
        torch.save(hd.net.state_dict(), model_filename)
        lock_filename.unlink(missing_ok=True)
    else:
        print(f"Model file is locked: {lock_filename}, passing.")

In [ ]:
with torch.no_grad():
    target_full = torch.nn.functional.pad(hd.target, (opt.N_pretime, opt.N_pretime))
    input_spikes = hd.get_input_spikes(p_A=hd.opt.p_A, N_pretime=hd.opt.N_pretime, N_trigger_time=hd.opt.num_delay, N_time=hd.opt.N_time)
    _, _, spikes = hd.forward_pass(input_spikes)
    spikes_evoked = spikes[:, :, (hd.opt.N_pretime+hd.opt.num_delay):(-hd.opt.N_pretime)]
    target_evoked = hd.target[:, :, hd.opt.num_delay:]

    precision, recall, f1_score = get_scores(spikes_evoked, target_evoked)
    print(f'precision = {precision:.3f}\t recall = {recall:.3f}\t f1_score = {f1_score:.3f} ')

In [ ]:
fig,ax = plt.subplots(figsize=(13, 8))
splt.raster(spikes[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="blue", marker='+', alpha=.5)
splt.raster(target_full[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, facecolor='none', edgecolor='red',  marker='o', alpha=.5)
splt.raster(input_spikes[i_SM, :opt.N_neuron_show, :opt.N_time_show].T, ax, s=25, c="green", marker='x', alpha=.5)
ax.vlines([opt.N_pretime], 0, opt.N_neuron_show, 'r', ls='--')
ax.vlines([opt.N_pretime+opt.num_delay], 0, opt.N_neuron_show, 'b', ls='--')
ax.set_xlabel("Time step (ms)")
ax.set_ylabel("Neuron Address")
ax.set_ylim(-1., opt.N_neuron_show + 1.)
if figpath is not None: printfig(fig, 'target', fig_width=opt.fig_width, fig_height=opt.fig_height)
spikes.shape, spikes[i_SM, :, :].sum().item(), hd.target[i_SM, :, :].sum().item()

In [ ]:
# hd.net.lin.bias.cpu().detach().numpy()
fig,ax = plt.subplots(figsize=(13, 8))
ax.hist(hd.net.lin.weight.cpu().detach().numpy().ravel(), bins=100, density=True) # pyright: ignore[reportCallIssue, reportAttributeAccessIssue]
ax.set_yscale('log')